# 04 - Single-line-to-ground fault

## Objective

Apply a declared single-line-to-ground fault and compare the solver-returned current from direct OpenDSS with the CEPT public CLI. Distinguish a calculated current from a protection-duty decision.

## Source, assumptions, and units

The source is the IEEE13 feeder bundled in the installed CEPT wheel. The fault is an SLG fault on phase 1 of bus `675` with `rf = 0.001 ohm`. Current is A, fault resistance is ohm, and voltage observations are pu. The source and fault settings are demonstrator assumptions, not field data or certified protection inputs.

## Prediction

The fault-element current magnitude should be positive and finite. Direct OpenDSS and CEPT should agree within the declared teaching tolerance because the fault type, bus, phase, resistance, and feeder source are matched.

## Action

Solve the same fault directly, then stream `cept study demo fault` into an exact run directory.

## Verification

Read phase currents from CEPT's persisted `results.json`, run `cept study verify`, and compare the total solver-returned current.

## Interpretation

The current depends on the declared source and feeder model. A public workflow receipt does not certify interrupting duty, relay settings, arc-flash analysis, or a field short-circuit result.

## Exercise

Change one explicit fault input in the direct command and in the lesson's declared prediction, such as `FAULT_RESISTANCE_OHM`. Explain how that change should affect current, then restart and rerun all cells.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide CEPT, OpenDSSDirect.py, and the bundled IEEE13 source files. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [1]:
# @title Setup — run once, then read the results below
import urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
_blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
assert hashlib.sha256(_blob).hexdigest() == _HELPER_SHA256, "lesson helper hash mismatch"
exec(compile(_blob, "lesson helper", "exec"))


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0
lesson helpers ready: cli/read/table/cards + WORKSPACE.


### Direct fault - solve without CEPT

One declared single-line-to-ground fault, solved raw. Convergence first, current second.


In [2]:
MASTER_DSS = ieee13_master()
FAULT_BUS = '675'
FAULT_PHASE = 1
FAULT_RESISTANCE_OHM = 0.001
import opendssdirect as dss

dss.Basic.ClearAll()
dss.Basic.DataPath(str(MASTER_DSS.parent))
dss.Text.Command(f'Redirect \"{MASTER_DSS}\"')
dss.Text.Command(f'New Fault.lesson_fault Bus1={FAULT_BUS}.{FAULT_PHASE} phases=1 r={FAULT_RESISTANCE_OHM}')
dss.Text.Command('Solve')
assert dss.Solution.Converged()


### Read the fault current back

Magnitude of the fault-element current, solver-returned in amperes. A zero here means the fault never engaged.


In [3]:
dss.Circuit.SetActiveElement('Fault.lesson_fault')
currents = dss.CktElement.Currents()
direct_current_a = abs(complex(currents[0], currents[1]))
table(['source', 'bus', 'phase', 'fault resistance', 'current', 'units'], [('direct OpenDSS', FAULT_BUS, FAULT_PHASE, FAULT_RESISTANCE_OHM, direct_current_a, 'ohm / A')])
assert direct_current_a > 0


| source | bus | phase | fault resistance | current | units |
| --- | --- | --- | --- | --- | --- |
| direct OpenDSS | 675 | 1 | 0.001 | 2950.698148621237 | ohm / A |


### Run the same fault through CEPT

`study run` persists solver artifacts under `runs/`; `study verify` checks the receipt. Both must pass before comparing.


In [4]:
RUN_DIR = WORKSPACE / 'runs' / '04-fault-study'
_run_summary_out = !cept study demo fault --network ieee13 --out {RUN_DIR} --force
run_summary = json.loads(" ".join(_run_summary_out))
_verify_summary_out = !cept study verify {RUN_DIR}
verify_summary = json.loads(" ".join(_verify_summary_out))


### Compare fault currents

`results.json` carries the solver-returned fault current. It must land within 1 A of the direct route; the cards repeat the verdict.


In [5]:
results = read(RUN_DIR / 'results.json')
fault = results['fault']
table(['source', 'bus', 'fault type', 'phase', 'fault resistance', 'current', 'units'], [('CEPT results.json', fault['bus'], fault['fault_type'], phase['phase'], fault['rf_ohm'], phase['i_amp'], 'ohm / A') for phase in fault['currents']])
cept_current_a = float(fault['total_fault_current_a'])
table(['source', 'total fault current', 'unit'], [('direct OpenDSS', direct_current_a, 'A'), ('CEPT results.json', cept_current_a, 'A')])

fault_abs_diff_a = abs(direct_current_a - cept_current_a)
cards([
    ('CEPT run', run_summary['status'], 'solver-backed run artifacts'),
    ('Verify', str(verify_summary['passed']), 'receipt integrity and convergence'),
    ('Total fault current', f'{cept_current_a:.1f} A', 'solver-returned current'),
    ('|direct − CEPT|', f'{fault_abs_diff_a:.2f} A', 'fault-current agreement'),
], title='4 · Fault-current agreement')
assert run_summary['status'] == 'PASS'
assert verify_summary['passed'] is True
assert fault['bus'].lower() == FAULT_BUS.lower() and fault['fault_type'] == 'slg'
assert fault['phases'] == [FAULT_PHASE]
assert abs(direct_current_a - cept_current_a) < 1.0


| source | bus | fault type | phase | fault resistance | current | units |
| --- | --- | --- | --- | --- | --- | --- |
| CEPT results.json | 675 | slg | 1 | 0.001 | 2950.7 | ohm / A |
| source | total fault current | unit |
| --- | --- | --- |
| direct OpenDSS | 2950.698148621237 | A |
| CEPT results.json | 2950.7 | A |


The displayed current is read from each solver route, and CEPT verification is performed from the exact persisted run. The result is a bounded `WORKFLOW_VALIDATED` demonstrator observation, not protection or project acceptance.